# Plot Constructors

Constructor helpers create ggpubr- and plotthis-style plots and return regular
`ggplot` objects. The examples therefore add scales and themes after
construction.


In [ ]:
import numpy as np
import pandas as pd
from plotnine_extra import *
from plotnine_extra.data import ToothGrowth, flights, iris, penguins

tooth = ToothGrowth.assign(dose=ToothGrowth["dose"].astype(str))
penguin_data = penguins.dropna(subset=[
    "bill_length_mm",
    "bill_depth_mm",
    "body_mass_g",
    "species",
]).copy()


## Distribution and group summaries

Use constructor arguments for the common mappings, then add scales and themes in
the standard plotnine way.


In [ ]:
(
    ggviolin(tooth, x="dose", y="len", fill="supp", add="boxplot")
    + scale_fill_tableau(k=2)
    + theme_pubr()
    + labs(x="Dose", y="Tooth length", fill="Supplement")
)


In [ ]:
(
    gghistogram(penguin_data, x="body_mass_g", fill="species", bins=24)
    + scale_fill_colorblind(k=3)
    + theme_clean()
    + labs(x="Body mass (g)", fill="Species")
)


## Scatter, bar, line, and paired plots

The constructor API stays small by design. For specialized plots, start with a
constructor and then add plotnine layers.


In [ ]:
(
    ggscatter(penguin_data, "bill_length_mm", "body_mass_g", color="species", add="reg.line")
    + scale_color_few(k=3)
    + theme_few()
)


In [ ]:
summary_data = tooth.groupby(["dose", "supp"], observed=True, as_index=False)["len"].mean()

(
    ggbarplot(summary_data, x="dose", y="len", fill="supp")
    + scale_fill_tableau(k=2)
    + theme_classic2()
    + labs(x="Dose", y="Mean tooth length", fill="Supplement")
)


In [ ]:
line_data = flights[flights["year"].isin([1950, 1951, 1952])].copy()
line_data["month_index"] = line_data.groupby("year").cumcount() + 1

(
    ggline(line_data, "month_index", "passengers", color="factor(year)")
    + scale_color_tableau(k=3)
    + theme_few()
    + labs(x="Month", y="Passengers", color="Year")
)


## Plotthis-style constructors

These helpers cover common bioinformatics plots without adding heavy optional
dependencies.


In [ ]:
rng = np.random.default_rng(7)
volcano = pd.DataFrame({
    "gene": [f"gene_{i}" for i in range(80)],
    "log2fc": rng.normal(0, 1.2, 80),
    "pvalue": np.clip(rng.beta(0.7, 8, 80), 1e-6, 1),
})

(
    ggvolcano(volcano, label="gene", label_top=5, p_cutoff=0.05, fc_cutoff=1.0)
    + theme_clean()
)


In [ ]:
embedding = pd.DataFrame({
    "UMAP_1": rng.normal(size=120),
    "UMAP_2": rng.normal(size=120),
    "cluster": rng.choice(["T", "B", "Mono"], size=120),
})
embedding["marker"] = np.exp(-((embedding["UMAP_1"] - 0.7) ** 2 + embedding["UMAP_2"] ** 2))

(
    ggfeaturedim(embedding, feature="marker")
    + theme_transparent()
    + labs(color="Marker")
)


In [ ]:
roc_data = pd.DataFrame({
    "truth": [0, 0, 0, 1, 1, 1, 1, 0],
    "score": [0.05, 0.2, 0.45, 0.55, 0.75, 0.8, 0.91, 0.35],
})

plot = ggroc(roc_data, truth="truth", score="score") + theme_tufte()
plot.data.attrs["auc"], plot


Aliases preserve familiar names from other plotting packages:
`VolcanoPlot`, `DimPlot`, `FeatureDimPlot`, and `ROCCurve`.
